## Section 1 — Setup

In [65]:
# Setting up imports and loading the cohort.
import sys
sys.path.append("..")
import json
from pathlib import Path
import pandas as pd
import numpy as np

cohort = pd.read_parquet("../data/cohort.parquet")
failed = cohort[cohort["is_failed"] == 1]
live = cohort[cohort["is_failed"] == 0]
print(len(failed), len(live))

1500 1500


In [66]:
# Importing the shared feature functions instead of defining them inline.
from src.features import (
    days_since_last_accounts,
    count_late_confirmation_statements,
    count_recent_resignations,
    count_new_charges,
    company_age_years,
    longest_filing_gap,
    filter_before_snapshot,
    get_failure_date,
    director_distress_count_safe,
    company_director_distress_safe,
)

## Section 2 — Failure dates and snapshot dates

### Determining failure dates and snapshot dates

In [67]:
# Computing the failure date and snapshot date for each failed company.
failure_dates = {}
for number in failed["CompanyNumber"]:
    data = json.loads((Path("../data/raw") / f"{number}.json").read_text())
    date = get_failure_date(data["filings"]["items"])
    if date:
        failure_dates[number] = date

failure_df = pd.DataFrame({
    "CompanyNumber": list(failure_dates.keys()),
    "failure_date": pd.to_datetime(list(failure_dates.values())),
})
failure_df["snapshot_date"] = failure_df["failure_date"] - pd.DateOffset(months=12)
len(failure_df)

1253

In [68]:
# Assigning each live company a snapshot date sampled from the failed companies' snapshot dates.
live = live.copy()
live["IncorporationDate"] = pd.to_datetime(live["IncorporationDate"], format="%d/%m/%Y")

rng = np.random.default_rng(1)
snapshot_pool = failure_df["snapshot_date"].to_numpy()

def sample_valid_snapshot(incorporation_date):
    valid = snapshot_pool[snapshot_pool > np.datetime64(incorporation_date)]
    if len(valid) == 0:
        return pd.NaT
    return rng.choice(valid)

live["snapshot_date"] = live["IncorporationDate"].apply(sample_valid_snapshot)
live_final = live.dropna(subset=["snapshot_date"]).copy()
len(live_final)

1266

## Section 3 — Building the features

### Feature 1: Days since accounts last filed

In [69]:
# First attempt: days overdue on accounts, reading the live profile directly.
# This version had a bug, kept here with the fix that follows, as a record
# of the investigation.
def days_overdue_on_accounts(profile: dict, snapshot_date: pd.Timestamp) -> float:
    """Return how many days overdue the company's accounts were, as of the
    snapshot date. Negative means not yet due. Missing due date returns NaN.
    """
    accounts = profile.get("accounts", {})
    due_on = accounts.get("next_accounts", {}).get("due_on")
    if not due_on:
        return float("nan")
    due_date = pd.to_datetime(due_on)
    return (snapshot_date - due_date).days

In [ ]:
# Checking company age for companies where the accounts due date is missing,
# using a larger sample of failed companies to investigate the pattern.
sample_check = []
for number in failed["CompanyNumber"].head(200):
    data = json.loads((Path("../data/raw") / f"{number}.json").read_text())
    snapshot = failure_df.loc[failure_df["CompanyNumber"] == number, "snapshot_date"]
    if len(snapshot) == 0:
        continue
    snapshot = snapshot.iloc[0]
    value = days_overdue_on_accounts(data["profile"], snapshot)
    sample_check.append({"CompanyNumber": number, "days_overdue": value})

sample_df = pd.DataFrame(sample_check)
sample_df["days_overdue"].describe()

count     166.000000
mean     -324.921687
std       745.602364
min     -4702.000000
25%      -543.250000
50%      -430.000000
75%      -279.250000
max      4214.000000
Name: days_overdue, dtype: float64

In [ ]:
# Testing the fixed feature on the same example company.
filtered_test = filter_before_snapshot(test_data, test_snapshot)
days_since_last_accounts(filtered_test["filings"], test_snapshot)

382

### Feature 2: Late confirmation statements

In [ ]:
# Checking whether filing descriptions ever mention lateness directly.
sample_descriptions = set()
for item in test_data["filings"]["items"]:
    sample_descriptions.add(item.get("description", ""))

sample_descriptions

{'accounts-amended-with-accounts-type-total-exemption-full',
 'accounts-with-accounts-type-micro-entity',
 'accounts-with-accounts-type-total-exemption-full',
 'change-account-reference-date-company-previous-extended',
 'change-registered-office-address-company-with-date-old-address-new-address',
 'confirmation-statement-with-no-updates',
 'confirmation-statement-with-updates',
 'gazette-filings-brought-up-to-date',
 'gazette-notice-compulsory',
 'liquidation-disclaimer-notice',
 'liquidation-voluntary-appointment-of-liquidator',
 'liquidation-voluntary-statement-of-affairs',
 'liquidation-voluntary-statement-of-receipts-and-payments-with-brought-down-date',
 'mortgage-create-with-deed-with-charge-number-charge-creation-date',
 'mortgage-satisfy-charge-full',
 'resolution'}

In [ ]:
# Looking at the gaps between consecutive confirmation statement filings.
confirmation_filings = sorted(
    [f["date"] for f in test_data["filings"]["items"] if f.get("type") == "CS01"]
)
dates = pd.to_datetime(confirmation_filings)
gaps = dates.to_series().diff().dropna()
gaps.dt.days

2021-04-19    384
2022-02-11    298
2022-11-30    292
2023-10-24    328
dtype: int64

In [ ]:
# Testing the feature on the same example company.
count_late_confirmation_statements(test_data["filings"]["items"])

1

### Feature 3: Recent director resignations

In [ ]:
# Looking at the structure of one officer record to see what fields are
# available for resignations.
test_data["officers"]["items"][0]

{'etag': 'c1e0e07d78ac5897e21e3fbd508f67b145fa6c0a',
 'address': {'address_line_1': 'Riverside 2, No.3, Campbell Road',
  'locality': 'Stoke On Trent',
  'postal_code': 'ST4 4RJ',
  'premises': 'C/O Currie Young Limited'},
 'appointed_on': '2016-01-29',
 'is_pre_1992_appointment': False,
 'country_of_residence': 'England',
 'date_of_birth': {'month': 9, 'year': 1971},
 'links': {'self': '/company/09977882/appointments/3vzn5oq3UYhx3PpUZ1DmK4CiAuM',
  'officer': {'appointments': '/officers/dBbjyr3J-1zlEohGF0H_G7DGesk/appointments'}},
 'name': 'LOGAN, Mara Maranda Mellissa',
 'nationality': 'British',
 'officer_role': 'director',
 'person_number': '204661170001',
 'identity_verification_details': {'appointment_verification_statement_due_on': '2025-11-18'}}

In [ ]:
# Finding an officer record that includes a resignation date, to confirm the field name.
for o in test_data["officers"]["items"]:
    if "resigned_on" in o:
        print(o["name"], o["resigned_on"])

TAYLOR, Jade Clairnese Minnet 2020-03-20
TAYLOR, Jade Clairnese Minnet 2018-02-14
TAYLOR, Keiran Martin 2019-04-16


In [ ]:
# Testing the feature on the same example company.
count_recent_resignations(test_data["officers"]["items"], test_snapshot)

0

### Feature 4: New charges registered

In [ ]:
# Looking at how charges are recorded in the filing history, to find the
# right filing type.
charge_related = set()
for item in test_data["filings"]["items"]:
    if "MR" in item.get("type", "") or "mortgage" in item.get("description", "").lower():
        charge_related.add((item["type"], item["description"]))

charge_related

{('MR01', 'mortgage-create-with-deed-with-charge-number-charge-creation-date'),
 ('MR04', 'mortgage-satisfy-charge-full')}

### Feature 5: Company age

In [ ]:
# Testing the feature on the same example company.
company_age_years(test_data["profile"], test_snapshot)

7.824777549623546

### Feature 6: Longest filing gap

In [ ]:
# Testing the feature on the same example company.
longest_filing_gap(test_data["filings"]["items"], test_snapshot)

287.0

## Section 4 — Assembling the features table

In [ ]:
# Building the features table for failed companies, using the leakage filter
# and all six features built above.
failed_rows = []
for _, row in failure_df.iterrows():
    number = row["CompanyNumber"]
    snapshot = row["snapshot_date"]
    data = json.loads((Path("../data/raw") / f"{number}.json").read_text())
    filtered = filter_before_snapshot(data, snapshot)

    failed_rows.append({
        "CompanyNumber": number,
        "snapshot_date": snapshot,
        "days_since_last_accounts": days_since_last_accounts(filtered["filings"], snapshot),
        "count_late_confirmation_statements": count_late_confirmation_statements(filtered["filings"]),
        "count_recent_resignations": count_recent_resignations(filtered["officers"], snapshot),
        "count_new_charges": count_new_charges(filtered["filings"], snapshot),
        "company_age_years": company_age_years(filtered["profile"], snapshot),
        "longest_filing_gap": longest_filing_gap(filtered["filings"], snapshot),
        "is_failed": 1,
    })

len(failed_rows)

1253

In [ ]:
# Building the features table for live companies.
live_rows = []
for _, row in live_final.iterrows():
    number = row["CompanyNumber"]
    snapshot = row["snapshot_date"]
    data = json.loads((Path("../data/raw") / f"{number}.json").read_text())
    filtered = filter_before_snapshot(data, snapshot)

    live_rows.append({
        "CompanyNumber": number,
        "snapshot_date": snapshot,
        "days_since_last_accounts": days_since_last_accounts(filtered["filings"], snapshot),
        "count_late_confirmation_statements": count_late_confirmation_statements(filtered["filings"]),
        "count_recent_resignations": count_recent_resignations(filtered["officers"], snapshot),
        "count_new_charges": count_new_charges(filtered["filings"], snapshot),
        "company_age_years": company_age_years(filtered["profile"], snapshot),
        "longest_filing_gap": longest_filing_gap(filtered["filings"], snapshot),
        "is_failed": 0,
    })

len(live_rows)

1266

In [ ]:
# Combining failed and live features into one table.
features_df = pd.DataFrame(failed_rows + live_rows)
features_df.shape

(2519, 9)

In [ ]:
# Excluding failed companies whose snapshot date falls before their
# incorporation date (companies that failed within 12 months of starting).
features_df = features_df[features_df["company_age_years"] >= 0].copy()
features_df.shape

(2511, 9)

In [ ]:
# Checking missing values across all features.
features_df.isna().sum()

CompanyNumber                           0
snapshot_date                           0
days_since_last_accounts              639
count_late_confirmation_statements      0
count_recent_resignations               0
count_new_charges                       0
company_age_years                       0
longest_filing_gap                    383
is_failed                               0
dtype: int64

In [ ]:
# Looking at the assembled table before any cleaning, to sanity check it.
features_df.head()

,CompanyNumber,snapshot_date,days_since_last_accounts,count_late_confirmation_statements,count_recent_resignations,count_new_charges,company_age_years,longest_filing_gap,is_failed
0,09977882,2023-11-26,382.0,1,0,1,7.824778,287.0,1
1,04544271,2021-10-31,NaN,0,0,0,19.101985,231.0,1
2,01287947,2024-10-08,NaN,0,0,0,47.868583,NaN,1
3,07161144,2025-06-26,177.0,1,0,0,15.353867,302.0,1
4,07682840,2021-12-17,331.0,1,0,0,10.475017,284.0,1


In [ ]:
# Excluding failed companies whose snapshot date falls before their
# incorporation date (companies that failed within 12 months of starting).
features_df = features_df[features_df["company_age_years"] >= 0].copy()
features_df.shape

(2511, 9)

In [ ]:
# Checking missing values across all features.
features_df.isna().sum()

CompanyNumber                           0
snapshot_date                           0
days_since_last_accounts              639
count_late_confirmation_statements      0
count_recent_resignations               0
count_new_charges                       0
company_age_years                       0
longest_filing_gap                    383
is_failed                               0
dtype: int64

In [ ]:
# Adding missingness flags before filling, so the model can learn from
# absence itself rather than a filled in value being mistaken for real data.
features_df["accounts_missing"] = features_df["days_since_last_accounts"].isna().astype(int)
features_df["filing_gap_missing"] = features_df["longest_filing_gap"].isna().astype(int)

features_df["days_since_last_accounts"] = features_df["days_since_last_accounts"].fillna(0)
features_df["longest_filing_gap"] = features_df["longest_filing_gap"].fillna(0)

features_df.isna().sum()

CompanyNumber                         0
snapshot_date                         0
days_since_last_accounts              0
count_late_confirmation_statements    0
count_recent_resignations             0
count_new_charges                     0
company_age_years                     0
longest_filing_gap                    0
is_failed                             0
accounts_missing                      0
filing_gap_missing                    0
dtype: int64

## Section 5 — Train/test split

In [ ]:
# Splitting into train and test by snapshot date, not randomly, so the
# split respects time order rather than letting the model see the future.
features_df = features_df.sort_values("snapshot_date")
split_index = int(len(features_df) * 0.8)
split_date = features_df.iloc[split_index]["snapshot_date"]

train = features_df[features_df["snapshot_date"] < split_date]
test = features_df[features_df["snapshot_date"] >= split_date]

print(len(train), len(test), split_date)

2008 503 2025-01-21 00:00:00


## Section 6 — Baseline

In [71]:
# Defining precision and recall at top k percent, since that matches how
# this model would actually be used - ranking companies, not a fixed cutoff.
def precision_at_k(y_true, y_scores, k_percent=10):
    """Return the precision among the top k percent highest scored companies."""
    n = int(len(y_true) * k_percent / 100)
    top_k_idx = np.argsort(y_scores)[-n:]
    return y_true.iloc[top_k_idx].mean()

def recall_at_k(y_true, y_scores, k_percent=10):
    """Return the share of all actual failures captured within the top k percent by score."""
    n = int(len(y_true) * k_percent / 100)
    top_k_idx = np.argsort(y_scores)[-n:]
    caught = y_true.iloc[top_k_idx].sum()
    total_failures = y_true.sum()
    return caught / total_failures

In [ ]:
# Scoring the baseline: ranking companies by days since accounts last filed,
# the single strongest individual feature.
baseline_precision_at_10 = precision_at_k(
    test["is_failed"].reset_index(drop=True),
    test["days_since_last_accounts"].reset_index(drop=True),
    k_percent=10
)
baseline_recall_at_10 = recall_at_k(
    test["is_failed"].reset_index(drop=True),
    test["days_since_last_accounts"].reset_index(drop=True)
)
print(baseline_precision_at_10, baseline_recall_at_10)

0.52 0.16149068322981366


## Section 7 — First model with 8 features

In [ ]:
# Importing xgboost for model training.
import xgboost as xgb

In [ ]:
# Training XGBoost on the leakage-safe features built so far.
feature_cols = [
    "days_since_last_accounts", "count_late_confirmation_statements",
    "count_recent_resignations", "count_new_charges", "company_age_years",
    "longest_filing_gap", "accounts_missing", "filing_gap_missing",
]

model = xgb.XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05)
model.fit(train[feature_cols], train["is_failed"])

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,True
,eval_metric,None


In [ ]:
# Scoring the model against the baseline.
model_scores = model.predict_proba(test[feature_cols])[:, 1]
model_precision_at_10 = precision_at_k(test["is_failed"].reset_index(drop=True), pd.Series(model_scores))
model_recall_at_10 = recall_at_k(test["is_failed"].reset_index(drop=True), pd.Series(model_scores))
print(model_precision_at_10, model_recall_at_10)

0.6 0.18633540372670807


## Feature 7: Director network score

In [ ]:
# Collecting director appointment histories for all directors of failed companies.
director_links = set()
for number in failed["CompanyNumber"]:
    data = json.loads((Path("../data/raw") / f"{number}.json").read_text())
    for o in data["officers"]["items"]:
        if o.get("officer_role") == "director":
            link = o.get("links", {}).get("officer", {}).get("appointments")
            if link:
                director_links.add(link)

len(director_links)

4426

In [ ]:
# Testing the director link collector on a small batch before the full run.
import sys
sys.path.append("..")
from src.collect_directors import collect_directors

test_batch = set(list(director_links)[:5])
collect_directors(test_batch)

finished. collected: 0, skipped: 5, failed: 0


In [ ]:
# Looking at one director's appointment history to confirm the structure.
sample_director_file = list(Path("../data/directors").glob("*.json"))[0]
sample_appointments = json.loads(sample_director_file.read_text())
sample_appointments.get("items", [])[0] if sample_appointments.get("items") else "no items"

{'address': {'address_line_1': 'Lightowlers Lane',
  'country': 'England',
  'locality': 'Littleborough',
  'postal_code': 'OL15 0LN',
  'premises': 'Langdale'},
 'appointed_on': '2024-09-07',
 'appointed_to': {'company_name': 'KIRBYS (WHITBY) LIMITED',
  'company_number': '01456618',
  'company_status': 'active'},
 'name': 'Richard Anthony WHIPP',
 'country_of_residence': 'England',
 'is_pre_1992_appointment': False,
 'links': {'company': '/company/01456618'},
 'name_elements': {'forename': 'Richard',
  'title': 'Mr',
  'other_forenames': 'Anthony',
  'surname': 'WHIPP'},
 'nationality': 'British',
 'officer_role': 'director',
 'identity_verification_details': {'appointment_verification_statement_due_on': '2026-09-16'}}

In [ ]:
# First attempt: counting a director's other companies by their current
# status. This version had a leakage bug, kept here with the investigation
# and fix that follow.
DISTRESS_STATUSES = {"liquidation", "administration", "receivership", "voluntary-arrangement"}

def director_distress_count(officer_link: str) -> int:
    """Count how many of a director's other companies show a distress status."""
    officer_id = officer_link.split("/")[2]
    path = Path("../data/directors") / f"{officer_id}.json"
    if not path.exists():
        return 0
    data = json.loads(path.read_text())
    count = 0
    for item in data.get("items", []):
        status = item.get("appointed_to", {}).get("company_status", "")
        if any(d in status.lower() for d in DISTRESS_STATUSES):
            count += 1
    return count

In [ ]:
# Testing the first attempt on one company from the failed group.
for o in test_data["officers"]["items"]:
    if o.get("officer_role") == "director":
        link = o.get("links", {}).get("officer", {}).get("appointments")
        if link:
            print(o["name"], director_distress_count(link))

LOGAN, Mara Maranda Mellissa 1
PARNELL - GURLING, Monique Yvonne 1
TAYLOR, Jade Clairnese Minnet 2
TAYLOR, Jade Clairnese Minnet 2
TAYLOR, Keiran Martin 1


In [ ]:
# Building the first (leaky) attempt across all failed and live companies.
def company_director_distress(officers: list) -> int:
    """Sum of distress counts across all directors of one company."""
    total = 0
    for o in officers:
        if o.get("officer_role") == "director":
            link = o.get("links", {}).get("officer", {}).get("appointments")
            if link:
                total += director_distress_count(link)
    return total

failed_director_scores = {}
for number in failed["CompanyNumber"]:
    data = json.loads((Path("../data/raw") / f"{number}.json").read_text())
    failed_director_scores[number] = company_director_distress(data["officers"]["items"])

live_director_scores = {}
for number in live_final["CompanyNumber"]:
    data = json.loads((Path("../data/raw") / f"{number}.json").read_text())
    live_director_scores[number] = company_director_distress(data["officers"]["items"])

len(failed_director_scores), len(live_director_scores)

(1500, 1266)

In [ ]:
# Building the first (leaky) attempt across all failed and live companies.
def company_director_distress(officers: list) -> int:
    """Sum of distress counts across all directors of one company."""
    total = 0
    for o in officers:
        if o.get("officer_role") == "director":
            link = o.get("links", {}).get("officer", {}).get("appointments")
            if link:
                total += director_distress_count(link)
    return total

failed_director_scores = {}
for number in failure_df["CompanyNumber"]:
    data = json.loads((Path("../data/raw") / f"{number}.json").read_text())
    failed_director_scores[number] = company_director_distress(data["officers"]["items"])

live_director_scores = {}
for number in live_final["CompanyNumber"]:
    data = json.loads((Path("../data/raw") / f"{number}.json").read_text())
    live_director_scores[number] = company_director_distress(data["officers"]["items"])

len(failed_director_scores), len(live_director_scores)

(1253, 1266)

In [ ]:
# Checking whether the director network feature is using today's status
# rather than status as of the snapshot date - this would be a leakage
# bug like Feature 1's original version.
sample_link = list(director_links)[0]
sample_officer_id = sample_link.split("/")[2]
sample_path = Path("../data/directors") / f"{sample_officer_id}.json"
sample = json.loads(sample_path.read_text())

for item in sample.get("items", [])[:5]:
    print(item.get("appointed_to", {}).get("company_number"), item.get("appointed_to", {}).get("company_status"), item.get("appointed_on"))

07792371 liquidation 2011-09-29


In [ ]:
# Narrowing to companies currently showing a distress status, since only
# these could ever contribute to the director distress signal.
distress_companies = set()
for path in Path("../data/directors").glob("*.json"):
    data = json.loads(path.read_text())
    for item in data.get("items", []):
        status = item.get("appointed_to", {}).get("company_status", "")
        number = item.get("appointed_to", {}).get("company_number")
        if number and any(d in status.lower() for d in DISTRESS_STATUSES):
            distress_companies.add(number)

len(distress_companies)

2576

In [ ]:
# Testing the linked filings collector on a small batch first.
from src.collect_director_links import collect_director_links

test_batch = set(list(distress_companies)[:5])
collect_director_links(test_batch)

finished. collected: 0, skipped: 5, failed: 0


In [ ]:
# Computing failure dates for the linked companies, using the same
# liquidation filing markers as before.
linked_failure_dates = {}
for path in Path("../data/director_links").glob("*.json"):
    number = path.stem
    data = json.loads(path.read_text())
    date = get_failure_date(data.get("items", []))
    if date:
        linked_failure_dates[number] = pd.to_datetime(date)

len(linked_failure_dates)

2077

In [ ]:
# Testing the corrected feature on the same example company.
company_director_distress_safe(test_data["officers"]["items"], test_snapshot, linked_failure_dates, Path("../data/directors"))

0

In [ ]:
# Building the safe director network feature across all failed companies.
failed_director_scores_safe = {}
for _, row in failure_df.iterrows():
    number = row["CompanyNumber"]
    snapshot = row["snapshot_date"]
    data = json.loads((Path("../data/raw") / f"{number}.json").read_text())
    failed_director_scores_safe[number] = company_director_distress_safe(data["officers"]["items"], snapshot, linked_failure_dates, Path("../data/directors"))
len(failed_director_scores_safe)

1253

In [ ]:
# Building the safe director network feature across all live companies.
live_director_scores_safe = {}
for _, row in live_final.iterrows():
    number = row["CompanyNumber"]
    snapshot = row["snapshot_date"]
    data = json.loads((Path("../data/raw") / f"{number}.json").read_text())
    live_director_scores_safe[number] = company_director_distress_safe(data["officers"]["items"], snapshot, linked_failure_dates, Path("../data/directors"))
len(live_director_scores_safe)

1266

In [ ]:
# Merging the safe director network scores into the features table.
director_scores_safe = {**failed_director_scores_safe, **live_director_scores_safe}
features_df["director_distress_score_safe"] = features_df["CompanyNumber"].map(director_scores_safe)
features_df["director_distress_score_safe"].describe()

count    2511.000000
mean        0.148148
std         1.804696
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max        60.000000
Name: director_distress_score_safe, dtype: float64

In [ ]:
# Redoing the split now that the director network feature has been added
# to features_df, so train and test both include the new column.
features_df = features_df.sort_values("snapshot_date")
split_index = int(len(features_df) * 0.8)
split_date = features_df.iloc[split_index]["snapshot_date"]

train = features_df[features_df["snapshot_date"] < split_date]
test = features_df[features_df["snapshot_date"] >= split_date]

print(len(train), len(test), split_date)

2008 503 2025-01-21 00:00:00


In [ ]:
# Retraining with the safe director network feature added, for the final
# 9 feature model.
feature_cols = [
    "days_since_last_accounts", "count_late_confirmation_statements",
    "count_recent_resignations", "count_new_charges", "company_age_years",
    "longest_filing_gap", "accounts_missing", "filing_gap_missing",
    "director_distress_score_safe",
]

model = xgb.XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05)
model.fit(train[feature_cols], train["is_failed"])

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,True
,eval_metric,None


In [ ]:
# Scoring the final model against the 8 feature result.
model_scores = model.predict_proba(test[feature_cols])[:, 1]
model_precision_at_10 = precision_at_k(test["is_failed"].reset_index(drop=True), pd.Series(model_scores))
model_recall_at_10 = recall_at_k(test["is_failed"].reset_index(drop=True), pd.Series(model_scores))
print(model_precision_at_10, model_recall_at_10)

0.68 0.2111801242236025


## Section 9 — Saving the final features table

In [ ]:
# Saving the final features table with all 9 features, including the
# director network score.
features_df.to_parquet("../data/features.parquet", index=False)
features_df.shape

(2511, 12)

## Section 10 — Saving the trained model

In [72]:
# Saving the trained model so it can be loaded by the API without retraining.
model.save_model("../data/model.json")